# 10 – Can Anything Beat the Market?

**Objective:** stop asking who will win, and ask where the bookmakers' prices are wrong.

**Key results**
- **Recent retirements are already in the price:** their coefficient is about zero, so the drop found in notebook 09 isn't a market error.
- **One small bias keeps repeating:** given the price, better-seeded players win slightly less. Of 63 variables, only this one survived, and it repeated on 2019–2021, 2022–2025 and the women's tour.
- **No proven profit at Pinnacle's odds:** the seed rule returned +7.09% and +4.40% for men, and −0.52% on 2,500 bets for women. All three intervals include zero.
- **Other bookmakers cost more:** every other bookmaker has a margin above 5%, which is why the same rule fires only 35 times in four years at Bet365.
- **The best lead is exploratory:** the seed rule at the best available price returned +4.31%, +7.04% and +2.83%, but we can't tell whether those prices could really be bet.

## 1. How this search was organised

Searching through many ideas guarantees that some will look good by chance, so each stage had a fixed role.

| Data | Role |
|---|---|
| Men, 2010–2018 | Search for ideas, and fix the rules |
| Men, 2019–2021 | First check of the rules that survived |
| Men, 2022–2025 | Second check. Our model was tested on these years in notebook 08, but this rule had never been applied to them |
| Women, 2007–2026 | A separate market, where no choice was ever made |

**Rules for keeping a variable:** it must pass a Benjamini-Hochberg correction at 5%, and have the same sign in both 2010–2014 and 2015–2018.

**Rules for bets:** flat stakes at Pinnacle's odds, with retirements refunded. A rule is only confirmed if its whole 95% interval is above zero.

In [1]:
import sqlite3
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")
sys.path.insert(0, "../src")
from tennis_data import load_all_matches
from tennis_domain_features import DOMAIN_GROUPS
from tennis_features import FEATURE_GROUPS
from tennis_market import (
    SEED_RULE, benjamini_hochberg, both_sides, edge, fair_probability, fit_on_market, load_wta_matches, margin,
    roi_summary, seed_rule_probability,
)
from tennis_models import INTERACTIONS, add_interactions
from tennis_odds import link_odds, load_odds

ODDS_DIR = Path("../data/raw/odds")
conn = sqlite3.connect(Path("../data/processed/tennis.db"))

ORDER = ["Elo", "Ranking", "Serve and return", "Form", "Fatigue and fitness", "Experience",
         "Style and age", "Head to head", "Draw", "Home"]
FINAL_COLUMNS = (
    [c for group in ORDER for c in FEATURE_GROUPS[group]]
    + list(INTERACTIONS) + DOMAIN_GROUPS["Tournament history"]
)

## 2. Which bookmakers could we beat?

The files contain many bookmakers, but most stopped being recorded years ago. From 2022, only Pinnacle, Bet365, the market average and the best available price remain.

Each bookmaker's margin is what a bettor pays on average. Accuracy and log loss use the probabilities left once the margin is removed.

In [2]:
BOOKS = {"PS": "Pinnacle", "B365": "Bet365", "EX": "Expekt", "LB": "Ladbrokes", "SJ": "Stan James", "UB": "Unibet",
         "CB": "Centrebet", "GB": "Gamebookers", "IW": "Interwetten", "SB": "Sportingbet", "B&W": "Bet&Win",
         "Avg": "Market average", "Max": "Best price"}
frames = []
for path in sorted(ODDS_DIR.iterdir()):
    if path.suffix in (".xls", ".xlsx") and not path.name.startswith("~$"):
        frames.append(pd.read_excel(path).assign(file_year=int(path.stem)))
all_odds = pd.concat(frames, ignore_index=True)
all_odds = all_odds[all_odds["Comment"] != "Walkover"]


def prices(code):
    winner = pd.to_numeric(all_odds[code + "W"], errors="coerce")
    loser = pd.to_numeric(all_odds[code + "L"], errors="coerce")
    total = 1 / winner.where(winner > 1) + 1 / loser.where(loser > 1)
    return winner, loser, total, total.between(0.9 if code == "Max" else 1.0, 1.25)


pinnacle_w, pinnacle_l, _, pinnacle_ok = prices("PS")
rows = []
for code, name in BOOKS.items():
    winner, loser, total, valid = prices(code)
    shared = valid & pinnacle_ok
    fair = fair_probability(winner[shared], loser[shared])
    fair_pinnacle = fair_probability(pinnacle_w[shared], pinnacle_l[shared])
    rows.append({
        "bookmaker": name, "first_year": int(all_odds.loc[valid, "file_year"].min()), "last_year": int(all_odds.loc[valid, "file_year"].max()),
        "matches": int(valid.sum()), "margin_pct": 100 * (total[valid].mean() - 1),
        "shared_with_pinnacle": int(shared.sum()),
        "log_loss_shared": -np.log(fair).mean() if shared.any() else np.nan,
        "pinnacle_log_loss_shared": -np.log(fair_pinnacle).mean() if shared.any() else np.nan,
        "pays_more_than_pinnacle_pct": 100 * (winner[shared] > pinnacle_w[shared]).mean() if shared.any() else np.nan,
    })
pd.DataFrame(rows).sort_values("margin_pct").round(4)

,bookmaker,first_year,last_year,matches,margin_pct,shared_with_pinnacle,log_loss_shared,pinnacle_log_loss_shared,pays_more_than_pinnacle_pct
12,Best price,2010,2026,41854,0.0731,39168,0.5742,0.5741,73.3481
0,Pinnacle,2004,2026,53953,2.3889,53953,0.5730,0.5730,0.0000
11,Market average,2010,2026,41997,5.7370,39302,0.5755,0.5745,8.0657
5,Unibet,2006,2009,10618,6.3217,7901,0.5688,0.5666,19.7317
1,Bet365,2002,2026,62655,6.5440,53446,0.5751,0.5738,13.9543
3,Ladbrokes,2008,2018,27919,6.6180,25192,0.5658,0.5632,20.1334
6,Centrebet,2001,2007,17451,6.6366,10537,0.5744,0.5738,20.7080
4,Stan James,2009,2014,15449,6.8013,12781,0.5594,0.5557,23.1046
2,Expekt,2004,2018,39185,6.9393,36357,0.5692,0.5664,17.5922
9,Sportingbet,2001,2003,5402,7.8009,0,NaN,NaN,NaN


**Finding:** no other bookmaker in the files looks easier to beat.

- **Margins:** Pinnacle charges 2.39% on average, while every other bookmaker charges more than 5%. So they must be far less accurate before they become worth betting against.
- **Accuracy:** on shared matches, each is a little less accurate than Pinnacle. For Bet365, the log loss is 0.5751 against 0.5738. That difference is small compared with the extra margin.
- **Prices:** each single bookmaker pays more than Pinnacle on the eventual winner in only about 14% to 23% of matches.
- **The exception:** the best price across bookmakers has a margin of just 0.07%, and it's almost as accurate as Pinnacle (0.5742 against 0.5741). Section 10 looks at it.

## 3. Preparing the men's data

Odds are linked to our matches from 2010 to 2025. The 2009 season has no Pinnacle odds, and 2026 has almost none.

Three new variables mark a player who retired within the last 1, 3 or 10 matches, following notebook 09.

In [3]:
data = pd.read_sql_query("SELECT * FROM final_features WHERE tourney_level <> 'D' AND year BETWEEN 2010 AND 2025", conn)
data = add_interactions(data)
keys = set(zip(data["tourney_id"], data["match_num"]))

matches = load_all_matches("../data/raw")
matches = matches[matches["year"] <= 2025]
played = matches[~matches["walkover"]]
retired = played["score"].astype(str).str.contains("RET", regex=False)
since, rows = {}, []
FAR = 10 ** 6
for tourney, number, winner, loser, ret in zip(played["tourney_id"], played["match_num"], played["winner_id"], played["loser_id"], retired):
    if (tourney, number) in keys:
        a, b = since.get(winner, FAR), since.get(loser, FAR)
        rows.append({"tourney_id": tourney, "match_num": number, **{f"d_ret_{k}": float(a < k) - float(b < k) for k in (1, 3, 10)}})
    # The result only updates the history after the match's features are read
    since[winner] = since.get(winner, FAR) + 1
    since[loser] = 0 if ret else since.get(loser, FAR) + 1
data = data.merge(pd.DataFrame(rows), on=["tourney_id", "match_num"], validate="one_to_one")

names = pd.read_sql_query("""
SELECT m.tourney_id, m.match_num, m.round, t.tourney_name, t.tourney_date,
       w.name_first AS winner_first, w.name_last AS winner_surname,
       l.name_first AS loser_first, l.name_last AS loser_surname
FROM matches AS m
JOIN tournaments AS t ON m.tourney_id = t.tourney_id
JOIN players AS w ON m.winner_id = w.player_id
JOIN players AS l ON m.loser_id = l.player_id
""", conn)
period = data[["tourney_id", "match_num"]].merge(names, on=["tourney_id", "match_num"], validate="one_to_one")
period["start_date"] = pd.to_datetime(period["tourney_date"])

odds = load_odds(ODDS_DIR, first_year=2010, last_year=2025)
period["odds_row"] = link_odds(period, odds)
print(f"Linked {period['odds_row'].notna().sum():,} of {len(period):,} matches")

RETIREMENT = ["d_ret_1", "d_ret_3", "d_ret_10"]
CANDIDATES = FINAL_COLUMNS + RETIREMENT
men = data.merge(period[["tourney_id", "match_num", "odds_row"]], on=["tourney_id", "match_num"], validate="one_to_one").dropna(subset=["odds_row"])
men = men.join(odds[["PSW", "PSL", "B365W", "B365L", "MaxW", "MaxL", "Comment", "Round"]], on="odds_row").reset_index(drop=True)
pinnacle = men.dropna(subset=["PSW", "PSL"]).reset_index(drop=True)
pinnacle["market_p"] = fair_probability(pinnacle["PSW"], pinnacle["PSL"])
pinnacle["logit_m"] = np.log(pinnacle["market_p"] / (1 - pinnacle["market_p"]))
pinnacle["z"] = pinnacle["d_seed"] / SEED_RULE["seed_sd"]
search = pinnacle[pinnacle["year"].between(2010, 2018)].reset_index(drop=True)
first_check = pinnacle[pinnacle["year"].between(2019, 2021)].reset_index(drop=True)
second_check = pinnacle[pinnacle["year"].between(2022, 2025)].reset_index(drop=True)
print(f"Search: {len(search):,}. First check: {len(first_check):,}. Second check: {len(second_check):,}.")

Linked 39,992 of 41,277 matches
Search: 23,104. First check: 6,193. Second check: 10,375.


## 4. Are Pinnacle's prices biased?

`logit(win) = a × logit(Pinnacle's probability)`

If `a` is 1, the price is right on average. If it's above 1, favourites win more often than the price suggests.

In [4]:
slam = search["tourney_level"] == "G"
groups = {"All": slam | ~slam, "2010–2014": search["year"] <= 2014, "2015–2018": search["year"] >= 2015,
          "Grand Slams": slam, "Masters": search["tourney_level"] == "M", "ATP 250 and 500": search["tourney_level"] == "A"}
rows = []
for name, mask in groups.items():
    fit = fit_on_market(search[mask], ["logit_m"])
    rows.append({"group": name, "matches": int(mask.sum()), "a": fit.params[0], "low_95": fit.conf_int()[0][0], "high_95": fit.conf_int()[0][1]})
pd.DataFrame(rows).round(3)

,group,matches,a,low_95,high_95
0,All,23104,1.051,1.019,1.083
1,2010–2014,12743,1.066,1.023,1.109
2,2015–2018,10361,1.032,0.984,1.081
3,Grand Slams,4473,1.132,1.065,1.200
4,Masters,4908,0.992,0.926,1.059
5,ATP 250 and 500,13589,1.035,0.991,1.079


In [5]:
sides = both_sides(search, "PSW", "PSL")
buckets = pd.cut(sides["fair"], np.arange(0, 1.01, 0.1))
rows = [{"fair_probability": str(bucket), "avg_odds": table["odds"].mean(), **roi_summary(table["profit"], rounds=2000)}
        for bucket, table in sides.groupby(buckets, observed=True)]
pd.DataFrame(rows).round(2)

,fair_probability,avg_odds,bets,profit_units,roi_pct,low_95,high_95
0,"(0.0, 0.1]",18.82,1728,-258.1,-14.94,-32.09,3.54
1,"(0.1, 0.2]",6.64,3508,-599.5,-17.09,-24.43,-9.87
2,"(0.2, 0.3]",3.92,5170,-501.3,-9.70,-14.14,-5.31
3,"(0.3, 0.4]",2.81,6073,-249.7,-4.11,-7.58,-0.78
4,"(0.4, 0.5]",2.19,5957,-163.6,-2.75,-5.47,0.06
5,"(0.5, 0.6]",1.76,5787,-110.9,-1.92,-4.20,0.27
6,"(0.6, 0.7]",1.50,6041,-106.5,-1.76,-3.50,0.04
7,"(0.7, 0.8]",1.31,5171,1.3,0.02,-1.50,1.53
8,"(0.8, 0.9]",1.15,3510,4.1,0.12,-1.17,1.44
9,"(0.9, 1.0]",1.04,1725,-26.5,-1.54,-2.63,-0.47


**Finding:** `a` is 1.051, and its interval excludes 1, so favourites are slightly underpriced overall. Long shots are clearly poor value: backing players with a fair chance of 10–20% returned −17.09%. Backing strong favourites (70–90%) is about break-even, so this bias alone doesn't beat the margin.

Grand Slams look most biased, with `a` at 1.132, though this was one of several groups examined. Section 6 checks it on new years.

## 5. Does any variable add to the price?

`logit(win) = a × logit(Pinnacle's probability) + β × variable`

If β isn't zero, the variable knows something the price doesn't. Each variable is tested on its own.

In [6]:
early, late = search["year"] <= 2014, search["year"] >= 2015


def half_beta(table):
    # A variable that never varies in this period can't be estimated, so its sign counts as unconfirmed
    if not table["x"].std() > 0:
        return np.nan
    return fit_on_market(table, ["logit_m", "x"]).params[1]


rows = []
for column in CANDIDATES:
    spread = search[column].std()
    if not spread > 0:
        continue
    table = search.assign(x=search[column] / spread)
    fit = fit_on_market(table, ["logit_m", "x"])
    rows.append({"variable": column, "beta": fit.params[1], "low_95": fit.conf_int()[1][0], "high_95": fit.conf_int()[1][1],
                 "p_value": fit.pvalues[1], "beta_2010_14": half_beta(table[early]), "beta_2015_18": half_beta(table[late])})
scan = pd.DataFrame(rows).sort_values("p_value").reset_index(drop=True)
scan["passes_correction"] = benjamini_hochberg(scan["p_value"])
scan["same_sign"] = np.sign(scan["beta_2010_14"]) == np.sign(scan["beta_2015_18"])
scan["survives"] = scan["passes_correction"] & scan["same_sign"]
print(f"{len(scan)} tests. Surviving both rules: {scan['survives'].sum()}. About {0.05 * len(scan):.1f} would reach p < 0.05 by chance alone.")
scan.head(10).round(4)

63 tests. Surviving both rules: 1. About 3.2 would reach p < 0.05 by chance alone.


,variable,beta,low_95,high_95,p_value,beta_2010_14,beta_2015_18,passes_correction,same_sign,survives
0,d_seed,-0.0714,-0.1111,-0.0318,0.0004,-0.0716,-0.0703,True,True,True
1,d_matches_14,0.0479,0.0182,0.0777,0.0016,0.0392,0.0589,False,True,False
2,d_matches_30,0.0456,0.0159,0.0753,0.0026,0.0416,0.0500,False,True,False
3,d_matches_7,0.0405,0.0105,0.0705,0.0081,0.0361,0.0460,False,True,False
4,d_vs_hand,-0.0394,-0.0691,-0.0097,0.0093,-0.0330,-0.0474,False,True,False
5,d_elo_x_slam,0.0658,0.0150,0.1167,0.0112,0.0493,0.0869,False,True,False
6,d_elo_x_best_of_5,0.0652,0.0144,0.1160,0.0119,0.0482,0.0869,False,True,False
7,d_minutes_14,0.0369,0.0070,0.0669,0.0157,0.0250,0.0516,False,True,False
8,d_log_rest,-0.0364,-0.0666,-0.0061,0.0184,-0.0314,-0.0416,False,True,False
9,d_recent_problem,0.0322,0.0028,0.0616,0.0316,0.0161,0.0542,False,True,False


In [7]:
scan[scan["variable"].isin(RETIREMENT)].round(4)

,variable,beta,low_95,high_95,p_value,beta_2010_14,beta_2015_18,passes_correction,same_sign,survives
54,d_ret_3,-0.0029,-0.0320,0.0262,0.8461,-0.0131,0.0108,False,False,False
58,d_ret_1,-0.0019,-0.0308,0.0270,0.8985,-0.0025,-0.0009,False,True,False
60,d_ret_10,0.0013,-0.0279,0.0306,0.9291,0.0009,0.0021,False,True,False


**Finding:** only `d_seed` survives both rules (β = −0.0714, p = 0.0004). Given the price, the player with the better seed wins slightly less often. Why isn't known: the market may overrate seeded players, but that's only a hypothesis.

**The three retirement variables are all about zero.** The drop after a retirement in notebook 09 is real, but the market seems to account for it.

## 6. The first check, 2019–2021

Two rules were frozen on 2010–2018, before these years were opened:

| Rule | Definition |
|---|---|
| **Slam favourites** | Back the favourite at a Grand Slam when its fair probability is 70–90% |
| **Seed rule** | `a` = 1.0995, `β` on `d_seed` = −0.0714, and bet when the edge is at least 2% |

The Slam range was chosen after seeing the search results, so its result there is optimistic.

In [8]:
fit = fit_on_market(search.assign(x=search["z"]), ["logit_m", "x"])
print(f"Fitted on the search years: a = {fit.params[0]:.4f}, beta = {fit.params[1]:.4f}, standard deviation of d_seed = {search['d_seed'].std():.4f}")
print("Frozen values:", SEED_RULE)


def slam_favourites(table):
    sides = both_sides(table, "PSW", "PSL")
    return sides[(sides["level"] == "G") & (sides["fair"] > 0.7) & (sides["fair"] <= 0.9)]


def seed_bets(table, winner_odds="PSW", loser_odds="PSL"):
    sides = both_sides(table.dropna(subset=[winner_odds, loser_odds]), winner_odds, loser_odds)
    return sides[sides["rule_edge"] >= SEED_RULE["threshold"]]


rows = []
for period_name, table in [("Search, 2010–2018", search), ("First check, 2019–2021", first_check)]:
    rows.append({"period": period_name, "rule": "Slam favourites", **roi_summary(slam_favourites(table)["profit"])})
    rows.append({"period": period_name, "rule": "Seed rule", **roi_summary(seed_bets(table)["profit"])})
pd.DataFrame(rows)

Fitted on the search years: a = 1.0995, beta = -0.0714, standard deviation of d_seed = 1.7923
Frozen values: {'a': 1.0995, 'beta': -0.0714, 'seed_sd': 1.7923, 'threshold': 0.02}


,period,rule,bets,profit_units,roi_pct,low_95,high_95
0,"Search, 2010–2018",Slam favourites,1860,48.8,2.62,0.57,4.62
1,"Search, 2010–2018",Seed rule,1295,54.6,4.21,-1.60,10.17
2,"First check, 2019–2021",Slam favourites,594,-3.9,-0.65,-4.68,3.20
3,"First check, 2019–2021",Seed rule,531,37.6,7.09,-2.16,16.36


In [9]:
fit = fit_on_market(first_check.assign(x=first_check["z"]), ["logit_m", "x"])
print(f"d_seed in 2019–2021: beta = {fit.params[1]:.4f} ({fit.conf_int()[1][0]:.4f}, {fit.conf_int()[1][1]:.4f}), p = {fit.pvalues[1]:.3f}")
fit = fit_on_market(first_check[first_check["tourney_level"] == "G"], ["logit_m"])
print(f"Slam bias in 2019–2021: a = {fit.params[0]:.3f} ({fit.conf_int()[0][0]:.3f}, {fit.conf_int()[0][1]:.3f})")

d_seed in 2019–2021: beta = -0.1014 (-0.1693, -0.0335), p = 0.003
Slam bias in 2019–2021: a = 1.048 (0.936, 1.160)


**Finding:**
- **Slam favourites failed.** The return fell to −0.65%, and `a` for Grand Slams fell to 1.048, with an interval that includes 1. The bias seen earlier was probably mostly noise.
- **The seed signal repeated.** `d_seed` is −0.1014 (p = 0.003).
- **The seed rule stayed positive, but isn't confirmed.** It returned +7.09% on 531 bets, and its interval includes zero.

## 7. The second check, 2022–2025

The seed rule was applied once more, unchanged.

In [10]:
fit = fit_on_market(second_check.assign(x=second_check["z"]), ["logit_m", "x"])
print(f"d_seed in 2022–2025: beta = {fit.params[1]:.4f} ({fit.conf_int()[1][0]:.4f}, {fit.conf_int()[1][1]:.4f}), p = {fit.pvalues[1]:.3f}")
bets = seed_bets(second_check)
print("Return by year (%):", (100 * bets.groupby("year")["profit"].mean()).round(2).to_dict())
pd.DataFrame([{"bookmaker": "Pinnacle", **roi_summary(bets["profit"])}])

d_seed in 2022–2025: beta = -0.0670 (-0.1226, -0.0114), p = 0.018
Return by year (%): {2022: 1.54, 2023: 4.05, 2024: 6.62, 2025: 5.53}


,bookmaker,bets,profit_units,roi_pct,low_95,high_95
0,Pinnacle,643,28.3,4.4,-4.04,12.62


In [11]:
# How often the same formula fires at a bookmaker with a higher margin
exploratory = men[men["year"].between(2022, 2025)].dropna(subset=["B365W", "B365L"])
print(f"Bet365, 2022–2025: {len(seed_bets(exploratory, 'B365W', 'B365L')):,} bets in four years")

Bet365, 2022–2025: 35 bets in four years


**Finding:** the signal repeats again (β = −0.0670, p = 0.018), and the rule returns +4.40% on 643 bets, positive in all four years. But its interval (−4.04% to +12.62%) includes zero, so **it isn't confirmed**. `d_seed` was part of the model tested on these years in notebook 08, although this rule had never been applied to them.

**The rule needs low margins.** At Bet365 it fires only 35 times in four years, too few to judge. The extra margin removes nearly every edge of 2% or more.

## 8. A separate market: women's tennis

The same rule, with every number taken from the men. Nothing is estimated on women's matches, and no choice was ever made here.

In [12]:
wta = load_wta_matches("../data/raw/wta")
wta_odds = load_odds(ODDS_DIR / "WTA")
wta["odds_row"] = link_odds(wta, wta_odds)
print(f"Linked {wta['odds_row'].notna().sum():,} of {len(wta):,} matches")
women = wta.dropna(subset=["odds_row"]).join(wta_odds[["PSW", "PSL", "Comment"]], on="odds_row").dropna(subset=["PSW", "PSL"]).reset_index(drop=True)
women["market_p"] = fair_probability(women["PSW"], women["PSL"])
women["logit_m"] = np.log(women["market_p"] / (1 - women["market_p"]))
women["z"] = women["d_seed"] / SEED_RULE["seed_sd"]
print(f"With Pinnacle odds: {len(women):,}, average margin {100 * margin(women['PSW'], women['PSL']).mean():.2f}%")

rows = []
for name, mask in [("All", women["year"] > 0), ("Up to 2018", women["year"] <= 2018),
                   ("2019–2021", women["year"].between(2019, 2021)), ("2022 onwards", women["year"] >= 2022)]:
    fit = fit_on_market(women[mask], ["logit_m", "z"])
    rows.append({"period": name, "matches": int(mask.sum()), "beta": fit.params[1], "low_95": fit.conf_int()[1][0],
                 "high_95": fit.conf_int()[1][1], "p_value": fit.pvalues[1]})
pd.DataFrame(rows).round(4)

Linked 44,311 of 47,199 matches
With Pinnacle odds: 40,458, average margin 2.51%


,period,matches,beta,low_95,high_95,p_value
0,All,40458,-0.0679,-0.0964,-0.0394,0.0000
1,Up to 2018,25377,-0.0628,-0.0996,-0.0259,0.0008
2,2019–2021,5550,-0.0745,-0.1482,-0.0008,0.0476
3,2022 onwards,9531,-0.0789,-0.1358,-0.0220,0.0066


In [13]:
women_bets = seed_bets(women)
print("Return by year (%):", (100 * women_bets.groupby("year")["profit"].mean()).round(1).to_dict())
pd.DataFrame([{"market": "Women, Pinnacle", **roi_summary(women_bets["profit"])}])

Return by year (%): {2007: 3.6, 2008: -5.2, 2010: 0.2, 2011: 3.4, 2012: 7.7, 2013: -6.6, 2014: -12.7, 2015: 8.7, 2016: -6.6, 2017: 5.1, 2018: -10.3, 2019: 0.0, 2020: -8.3, 2021: 6.7, 2022: 13.8, 2023: 15.1, 2024: -18.5, 2025: -4.9, 2026: -70.0}


,market,bets,profit_units,roi_pct,low_95,high_95
0,"Women, Pinnacle",2500,-13.0,-0.52,-4.74,3.75


**Finding:** this is the most informative test.

- **The bias is real.** The coefficient is −0.0679, almost the same as the men's −0.0714, on a market where no choice was ever made. It holds in every period.
- **The profit isn't there.** The rule returned −0.52% on 2,500 bets (−4.74% to +3.75%).

**A biased price isn't the same as a profit.** The effect is real but small, and Pinnacle's margin of about 2.5% seems to absorb it. Looking back, the men's +7.09% and +4.40% were probably helped by luck.

## 9. Could many small signals add up?

One-at-a-time tests can miss small signals that matter together. This LASSO starts from Pinnacle's price and adds every variable at once.

**The rule fixed in advance:** open 2019–2021 only if the log loss beats Pinnacle with its whole interval above zero, and the return is positive. Each year from 2013 is predicted using earlier years only.

In [14]:
PRICE_SCALE = 100  # A large scale makes the penalty on the market's coefficient negligible
TEST_YEARS = range(2013, 2019)
usable = [c for c in CANDIDATES if search[c].std() > 0]


def matrix(table, spread):
    x = (table[usable] / spread).fillna(0).to_numpy()
    return np.column_stack([table["logit_m"].to_numpy() * PRICE_SCALE, x])


def walk_forward_on_market(C):
    predictions = pd.Series(np.nan, index=search.index)
    for year in TEST_YEARS:
        train, current = search[search["year"] < year], search[search["year"] == year]
        spread = train[usable].std().replace(0, 1)
        x = matrix(train, spread)
        model = LogisticRegression(penalty="l1", solver="liblinear", C=C, fit_intercept=False, max_iter=500, random_state=0)
        model.fit(np.vstack([x, -x]), np.r_[np.ones(len(x)), np.zeros(len(x))])
        predictions[current.index] = model.predict_proba(matrix(current, spread))[:, 1]
    return predictions, int((model.coef_[0][1:] != 0).sum())


def log_losses(p):
    return -np.log(np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6))


tested = search["year"].isin(TEST_YEARS)
market_loss = log_losses(search.loc[tested, "market_p"])
rng = np.random.default_rng(0)
rows, stored = [], {}
for C in [0.001, 0.003, 0.01, 0.03, 0.1]:
    predictions, kept = walk_forward_on_market(C)
    stored[C] = predictions
    gain = market_loss - log_losses(predictions[tested])
    draws = [gain[rng.integers(0, len(gain), len(gain))].mean() for _ in range(2000)]
    rows.append({"C": C, "variables_kept": kept, "gain_in_log_loss": gain.mean(),
                 "low_95": np.percentile(draws, 2.5), "high_95": np.percentile(draws, 97.5)})
pd.DataFrame(rows).round(5)

,C,variables_kept,gain_in_log_loss,low_95,high_95
0,0.001,0,0.00006,-0.00035,0.00046
1,0.003,1,0.00006,-0.00035,0.00048
2,0.010,20,0.00032,-0.00022,0.00086
3,0.030,40,0.00029,-0.00070,0.00128
4,0.100,52,-0.00035,-0.00163,0.00091


In [15]:
best_C = max(stored, key=lambda C: (market_loss - log_losses(stored[C][tested])).mean())
table = search[tested].assign(p=stored[best_C][tested])
sides = pd.concat([
    pd.DataFrame({"p": table["p"], "odds": table["PSW"], "won": 1.0, "comment": table["Comment"]}),
    pd.DataFrame({"p": 1 - table["p"], "odds": table["PSL"], "won": 0.0, "comment": table["Comment"]}),
], ignore_index=True)
sides = sides[sides["comment"] == "Completed"].copy()
sides["profit"] = np.where(sides["won"] == 1, sides["odds"] - 1, -1.0)
sides["edge"] = edge(sides["p"], sides["odds"])
pd.DataFrame([{"edge_at_least": t, **roi_summary(sides.loc[sides["edge"] >= t, "profit"], rounds=2000)} for t in [0, 0.01, 0.02, 0.03, 0.05]])

,edge_at_least,bets,profit_units,roi_pct,low_95,high_95
0,0.00,3923,93.9,2.39,-0.10,5.10
1,0.01,1538,35.7,2.32,-2.64,7.15
2,0.02,653,56.4,8.64,0.33,17.11
3,0.03,296,20.8,7.03,-5.26,19.60
4,0.05,52,1.8,3.50,-29.85,39.52


**Finding:** this fails the rule set in advance. The best gain in log loss is +0.00032, and its interval includes zero, so 2019–2021 wasn't opened for this model.

The +8.64% return at an edge of 2% looks tempting, but it's the best of several thresholds, and it doesn't rise steadily as the edge grows. It's weak evidence at best.

## 10. Exploratory: the best available price

The bookmakers table shows why this deserves a look: the best price across bookmakers has almost no margin. The seed rule's probability comes from Pinnacle, and the bet is placed at the best price.

**This can't be used to claim a profit.** It's a new idea, tried after seeing everything above, on years already used. The best price may also have been available only briefly, or only for small stakes.

In [16]:
best = pinnacle.dropna(subset=["MaxW", "MaxL"]).reset_index(drop=True)
best = best[best["year"] >= 2011].reset_index(drop=True)
fair = fair_probability(best["PSW"], best["PSL"])
common = {"comment": best["Comment"], "year": best["year"]}
sides = pd.concat([
    pd.DataFrame({"p": seed_rule_probability(fair, best["d_seed"]), "odds": best["MaxW"], "won": 1.0, **common}),
    pd.DataFrame({"p": seed_rule_probability(1 - fair, -best["d_seed"]), "odds": best["MaxL"], "won": 0.0, **common}),
], ignore_index=True)
sides = sides[sides["comment"] == "Completed"].copy()
sides["profit"] = np.where(sides["won"] == 1, sides["odds"] - 1, -1.0)
sides["edge"] = edge(sides["p"], sides["odds"])
rows = []
for name, mask in [("2011–2018", sides["year"] <= 2018), ("2019–2021", sides["year"].between(2019, 2021)), ("2022–2025", sides["year"] >= 2022)]:
    rows.append({"period": name, **roi_summary(sides.loc[mask & (sides["edge"] >= SEED_RULE["threshold"]), "profit"], rounds=5000)})
pd.DataFrame(rows)

,period,bets,profit_units,roi_pct,low_95,high_95
0,2011–2018,10558,455.1,4.31,1.31,7.57
1,2019–2021,3720,261.9,7.04,2.78,11.41
2,2022–2025,4270,121.0,2.83,-0.96,6.71


**Two checks.** Does this depend on the seed rule at all? Here, a bet is any best price at least 2% above Pinnacle's fair price, with no seeds involved. The second table shows the return by how far the price is above fair: very large gaps are the ones most likely to be stale or errors.

In [17]:
sides["pure_edge"] = edge(pd.concat([fair, 1 - fair], ignore_index=True).reindex(sides.index), sides["odds"])
rows = []
for name, mask in [("2011–2018", sides["year"] <= 2018), ("2019–2021", sides["year"].between(2019, 2021)), ("2022–2025", sides["year"] >= 2022)]:
    chosen = sides[mask & (sides["pure_edge"] >= 0.02)]
    rows.append({"period": name, "avg_odds": chosen["odds"].mean(), **roi_summary(chosen["profit"], rounds=5000)})
pd.DataFrame(rows).round(2)

,period,avg_odds,bets,profit_units,roi_pct,low_95,high_95
0,2011–2018,13.41,10255,93.5,0.91,-3.24,5.03
1,2019–2021,4.24,3500,191.5,5.47,-0.09,11.20
2,2022–2025,4.14,3767,-19.1,-0.51,-5.70,4.80


In [18]:
gaps = pd.cut(sides["pure_edge"], [-1, 0, 0.02, 0.05, 0.10, 0.20, 100], labels=["Below fair", "0–2%", "2–5%", "5–10%", "10–20%", "Above 20%"])
rows = [{"price_above_fair": gap, "avg_odds": table["odds"].mean(), **roi_summary(table["profit"], rounds=2000)}
        for gap, table in sides.groupby(gaps, observed=True)]
pd.DataFrame(rows).round(2)

,price_above_fair,avg_odds,bets,profit_units,roi_pct,low_95,high_95
0,Below fair,2.62,41629,-1073.5,-2.58,-3.78,-1.45
1,0–2%,2.49,12745,69.1,0.54,-1.38,2.61
2,2–5%,3.09,8996,39.0,0.43,-2.40,3.38
3,5–10%,4.15,5192,13.5,0.26,-4.39,4.69
4,10–20%,7.15,2359,26.2,1.11,-8.03,10.62
5,Above 20%,17.38,973,189.3,19.45,-9.30,52.61


**Finding:** this is the most promising result, and also the least certain.

- **With the seed rule, all three periods are positive:** +4.31%, +7.04% and +2.83%, over about 18,500 bets. The first two intervals are above zero, but the most recent one includes it (−0.96% to +6.71%).
- **Without the seed rule, the result mostly disappears:** +0.91%, +5.47% and −0.51%, with all three intervals including zero. So simply choosing the highest price isn't enough.
- **Prices far above fair are mostly noise.** Between 0% and 20% above fair, the return is close to zero. The prices above 20% are rare and have average odds above 17, and their interval is very wide.

**Why this isn't a result to rely on:**
- The idea came after everything above, using years already examined.
- The rule's numbers were fitted on 2010–2018, so the first row isn't an independent test.
- The file only gives the best price, not when it was offered, for how much, or whether Pinnacle's price was from the same moment.
- Bookmakers can limit customers who keep taking their best prices.

**The only way to settle it is a test on future matches,** recording prices that could really be bet. The rule fires about 1,000 times a year at the best price, against about 160 at Pinnacle, so this test would be faster.

## 11. Conclusions

| Question | Answer |
|---|---|
| Is another bookmaker easier to beat? | No: all charge much higher margins, and none is much less accurate |
| Does the market ignore retirements? | No: they're already in the price |
| Is any bias real? | Yes: the seed effect repeats on new years and on the women's tour |
| Does it make money at Pinnacle's odds? | Not proven: +4.40% for men and −0.52% for women, with intervals including zero |
| Does it work at the best price? | Unknown: positive in the past, but we can't tell if those prices could be bet |
| Should real money be used? | No. Nothing here passes the test set in advance |

**Limits:** these look like prices close to the start of the match, and we don't know how much could be staked at them. Trying many ideas also weakens each positive result, even with these precautions.

**Next step:** the app's Paper trading tab records this rule on future matches, with no real money.